In [8]:
from datetime import date, timedelta
import pandas as pd
import plotly.graph_objects as go
from bcb import sgs
import numpy as np

In [15]:
# A API SGS limita séries diárias a consultas de no máximo 10 anos.
inicio = date(2010, 11, 27)
fim = date.today()
janelas = []

while inicio <= fim:
    fim_janela = min(inicio + timedelta(days=3650), fim)
    janelas.append(
        sgs.get(
            {"dolar_venda": 1},
            start=inicio.isoformat(),
            end=fim_janela.isoformat(),
        )
    )
    inicio = fim_janela + timedelta(days=1)

f = pd.concat(janelas).sort_index()
serie = f.iloc[:, 0].dropna()

fig = go.Figure(
    go.Scatter(
        x=serie.index,
        y=serie.values,
        mode="lines",
        name="Valor",
        line=dict(color="#2563eb", width=2.5),
        hovertemplate="%{x|%d/%m/%Y}<br>Valor: %{y:.4f}<extra></extra>",
    )
)

valor_atual = serie.iloc[-1]
data_atual = serie.index[-1]

fig.add_trace(
    go.Scatter(
        x=[data_atual],
        y=[valor_atual],
        mode="markers+text",
        text=[f"{valor_atual:.2f}"],  # use f"R$ {valor_atual:.4f}" para exibir a moeda
        textposition="middle right",
        marker=dict(color="#dc2626", size=9),
        showlegend=False,
    )
)

fig.update_layout(showlegend=False)

fig.update_layout(
    title="Taxa de câmbio - Livre - Dólar americano (venda)",
    xaxis_title="Data",
    yaxis_title="Valor (R$)",
    template="plotly_white",
    hovermode="x unified",
    height=500,
)

fig.update_xaxes(
    rangeselector=dict(
        buttons=[
            dict(count=1, label="1 Ano", step="year", stepmode="backward"),
            dict(count=3, label="3 Anos", step="year", stepmode="backward"),
            dict(count=5, label="5 Anos", step="year", stepmode="backward"),
            dict(label="Tudo", step="all"),
        ]
    ),
    rangeslider_visible=False
)

fig.show()

In [16]:
# ==================================================
# IPCA Mensal
# ==================================================

inicio = date(2018, 1, 1)
fim = date.today()

janelas = []

while inicio <= fim:

    fim_janela = min(
        inicio + timedelta(days=365),
        fim
    )

    janelas.append(
        sgs.get(
            {"IPCA": 433},
            start=inicio.isoformat(),
            end=fim_janela.isoformat()
        )
    )

    inicio = fim_janela + timedelta(days=1)

df = pd.concat(janelas).sort_index()

serie_ipca = df.iloc[:, 0].dropna()

# ==================================================
# IPCA acumulado 12 meses
# ==================================================

ipca_12m = (
    (1 + serie_ipca / 100)
    .rolling(12)
    .apply(np.prod, raw=True)
    .sub(1)
    .mul(100)
)

# ==================================================
# Último valor
# ==================================================

ipca_12m_valid = ipca_12m.dropna()

ultimo_valor = ipca_12m_valid.iloc[-1]
ultima_data = ipca_12m_valid.index[-1]

# ==================================================
# Figura
# ==================================================

fig = go.Figure()

# IPCA mensal

fig.add_trace(
    go.Scatter(
        x=serie_ipca.index,
        y=serie_ipca,
        mode="lines",
        name="IPCA Mensal",
        line=dict(
            color="#2563eb",
            width=2
        ),
        hovertemplate=
        "<b>%{x|%m/%Y}</b><br>"
        "IPCA Mensal: %{y:.2f}%"
        "<extra></extra>",
    )
)

# IPCA acumulado 12 meses

fig.add_trace(
    go.Scatter(
        x=ipca_12m.index,
        y=ipca_12m,
        mode="lines",
        name="IPCA Acumulado 12M",
        line=dict(
            color="#dc2626",
            width=3
        ),
        hovertemplate=
        "<b>%{x|%m/%Y}</b><br>"
        "IPCA 12M: %{y:.2f}%"
        "<extra></extra>",
    )
)

# marcador final

fig.add_trace(
    go.Scatter(
        x=[ultima_data],
        y=[ultimo_valor],
        mode="markers",
        marker=dict(
            color="#dc2626",
            size=9
        ),
        showlegend=False,
        hovertemplate=
        "<b>%{x|%m/%Y}</b><br>"
        "IPCA 12M: %{y:.2f}%"
        "<extra></extra>",
    )
)

# ==================================================
# Anotação do último valor
# ==================================================

fig.add_annotation(
    x=ultima_data,
    y=ultimo_valor,
    text=f"<b>{ultimo_valor:.2f}%</b>",
    showarrow=False,
    xshift=30,
    font=dict(
        size=12,
        color="#dc2626"
    )
)

# ==================================================
# Meta de inflação
# ==================================================

# fig.add_hrect(
#     y0=1.5,
#     y1=4.5,
#     fillcolor="green",
#     opacity=0.08,
#     layer="below",
#     line_width=0,
# )

# fig.add_hline(
#     y=3.0,
#     line_color="green",
#     line_dash="dash",
#     annotation_text="Meta (3,0%)"
# )

# fig.add_hline(
#     y=4.5,
#     line_color="orange",
#     line_dash="dot",
#     annotation_text="Limite Superior (4,5%)"
# )

# fig.add_hline(
#     y=1.5,
#     line_color="orange",
#     line_dash="dot",
#     annotation_text="Limite Inferior (1,5%)"
# )
# Linhas horizontais

fig.add_hline(
    y=3.0,
    line_color="green",
    line_dash="dash"
)

fig.add_hline(
    y=4.5,
    line_color="orange",
    line_dash="dot"
)

fig.add_hline(
    y=1.5,
    line_color="orange",
    line_dash="dot"
)

# Labels à esquerda

fig.add_annotation(
    x=serie_ipca.index.min(),
    y=4.5,
    text="Limite Superior (4,5%)",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color="orange")
)

fig.add_annotation(
    x=serie_ipca.index.min(),
    y=3.0,
    text="Meta (3,0%)",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color="green")
)

fig.add_annotation(
    x=serie_ipca.index.min(),
    y=1.5,
    text="Limite Inferior (1,5%)",
    showarrow=False,
    xanchor="left",
    yanchor="bottom",
    font=dict(color="orange")
)

# ==================================================
# Layout
# ==================================================

fig.update_layout(
    title="IPCA Mensal x IPCA Acumulado em 12 Meses",
    template="plotly_white",
    hovermode="x unified",
    height=600,
    xaxis_title="Data",
    yaxis_title="Variação (%)",
    legend=dict(
        orientation="h",
        y=1.08,
        x=0.5,
        xanchor="center"
    ),
)

# ==================================================
# Navegação
# ==================================================


fig.update_xaxes(
    rangeselector=dict(
        buttons=[
            dict(count=1, label="1 Ano", step="year", stepmode="backward"),
            dict(count=3, label="3 Anos", step="year", stepmode="backward"),
            dict(count=5, label="5 Anos", step="year", stepmode="backward"),
            dict(label="Tudo", step="all"),
        ]
    ),
    rangeslider_visible=False
)

fig.show()